# Step 05 — Resume Parser fine-tuning on Kaggle (both T4s)

Use a fresh **GPU T4 x2** session with **Internet enabled**. Enable the named secrets under Add-ons → Secrets before **Save Version → Save & Run All**. W&B tracking and post-evaluation are enabled; the evaluated adapter publishes to `md-zain/resume-parser` on `main`.

Train for **one epoch over all retained clean training records**. There is no sample cap, positive step override or notebook time budget. Complete training and validation examples exceeding the model's native context are filtered before weights are loaded, without truncating target JSON. A dataset-summary cell displays source, cleanup, overlength exclusion and final usage counts for both splits. Duplicate/conflicting records and held-out overlap remain excluded; original source files are preserved.

The notebook launches one 4-bit Unsloth/FP16 training process per GPU. Batch size stays 1 per GPU; accumulation preserves an effective batch of 16. Full validation loss runs before training, every 50 optimizer steps alongside checkpoints, and at the final step. Full JSON extraction scoring runs on every retained validation resume after training. This avoids repeating slow JSON generation at every checkpoint.

Source, secrets, dataset files and prepared training inputs stay under `/tmp`. Checkpoints, the adapter, anonymous evaluation tables and per-GPU memory measurements are saved under `/kaggle/working`. Hugging Face checkpoint commits support recovery if Kaggle interrupts the session. Hardware memory, the model's native context window and Kaggle's session/quota limits still apply; native-context exclusions are counted explicitly; memory fit must still be verified on Kaggle.


In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"
# Cell 1 — Expose both Kaggle T4s before importing the training stack.
import subprocess
import sys
import time  # Used only for network retries, not a training time limit.
from pathlib import Path

if not Path('/kaggle/working').is_dir():
    raise RuntimeError('Run this notebook in Kaggle with a GPU accelerator enabled.')
if any(name in sys.modules for name in ('torch', 'unsloth', 'transformers', 'peft', 'trl')):
    raise RuntimeError('Start a fresh Kaggle session and run from the first cell before importing the training stack.')
if int(os.getenv('WORLD_SIZE', '1')) != 1:
    raise RuntimeError('Use the normal single-process notebook kernel, not a distributed launcher.')
PRIVATE_ROOT = Path('/tmp/resume-parser-kaggle')
PRIVATE_ROOT.mkdir(parents=True, exist_ok=True)
PRIVATE_ROOT.chmod(0o700)
os.environ['HF_HOME'] = str(PRIVATE_ROOT / 'huggingface')
os.environ['HF_TOKEN_PATH'] = str(PRIVATE_ROOT / 'huggingface/token')
os.environ['WANDB_DIR'] = str(PRIVATE_ROOT / 'wandb')
Path(os.environ['WANDB_DIR']).mkdir(parents=True, exist_ok=True)
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
# nvidia-smi lists physical devices; the child-process check confirms CUDA visibility.
subprocess.run(['nvidia-smi'], check=True)
subprocess.run([sys.executable, '-c', (
    'import torch; '
    'print("torch.cuda.device_count():", torch.cuda.device_count()); '
    'assert torch.cuda.is_available() and torch.cuda.device_count() >= 1, '
    '"Expected visible CUDA GPUs; start a fresh GPU session."; '
    'print("Visible GPU:", torch.cuda.get_device_name(0))'
)], check=True)


In [ ]:
# Cell 2 — Read enabled Kaggle Secrets; never print their values.
from kaggle_secrets import UserSecretsClient
s = UserSecretsClient()
# [LIST OF ENV VARIABLE NAMES] — add optional project keys here.
ENV_VARIABLE_NAMES = [
    'HF_TOKEN', 'GITHUB_TOKEN', 'HF_DATASET_REVISION',
    'WANDB_API_KEY', 'WANDB_ENTITY', 'WANDB_PROJECT', 'TYPESAFE_API_KEY',
]
try:
    hf_token = s.get_secret('HF_TOKEN').strip()
except Exception:
    raise RuntimeError('Enable a write-capable HF_TOKEN in Kaggle Add-ons → Secrets for this notebook.') from None
if not hf_token:
    raise RuntimeError('HF_TOKEN is empty. Update the Kaggle secret before running.')
os.environ['HF_TOKEN'] = hf_token
loaded_secret_names = {'HF_TOKEN'}
for name in ENV_VARIABLE_NAMES:
    if name == 'HF_TOKEN':
        continue
    try:
        value = s.get_secret(name)
    except Exception:
        continue
    if value:
        os.environ[name] = value
        loaded_secret_names.add(name)
print('Enabled secret names:', ', '.join(sorted(loaded_secret_names)))


In [ ]:
# Cell 3 — Clone public/private GitHub source, or copy an attached Kaggle source folder.
import base64
import shutil
import socket
from urllib.parse import urlsplit

REPO_URL = 'https://github.com/Muhammad-Zain01/resume-parser-model.git'  # [REPO_URL]
SOURCE_DIR = None  # Optional: '/kaggle/input/YOUR-SOURCE-DATASET/resume-parser-model'
PROJECT_ROOT = PRIVATE_ROOT / 'source'
MARKER = Path('src/steps/05_fine_tuning/fine_tuning.py')

def check_dns(host):
    for attempt in range(3):
        try:
            socket.getaddrinfo(host, 443)
            return
        except socket.gaierror:
            if attempt < 2:
                time.sleep(2)
    raise RuntimeError(
        f'Cannot resolve {host}. Enable Internet in Kaggle notebook settings and save those '
        'settings before Save & Run All. If Internet is already enabled, restart the session '
        'and retry; this is a network/DNS problem. For GitHub-only outages, attach a source '
        'folder as Kaggle input and set SOURCE_DIR above. Pip and Hugging Face still need Internet.'
    )

if SOURCE_DIR:
    source = Path(SOURCE_DIR).resolve()
    if not (source / MARKER).is_file():
        raise FileNotFoundError('SOURCE_DIR must contain src/steps/05_fine_tuning/fine_tuning.py.')
    if PROJECT_ROOT.exists():
        raise RuntimeError('Source is already staged. Start a fresh session before changing SOURCE_DIR.')
    shutil.copytree(source, PROJECT_ROOT, ignore=shutil.ignore_patterns(
        '.git', '.env', '.env.*', '.data', '.venv', '__pycache__', '*.ipynb_checkpoints',
    ))
else:
    parsed = urlsplit(REPO_URL)
    if (parsed.scheme != 'https' or parsed.hostname != 'github.com' or parsed.username
            or parsed.password or parsed.query or parsed.fragment or parsed.port):
        raise ValueError('Use a clean https://github.com/OWNER/REPO.git URL without credentials or query parameters.')
    check_dns('github.com')
    git_env = os.environ.copy()
    git_env['GIT_TERMINAL_PROMPT'] = '0'
    github_token = os.getenv('GITHUB_TOKEN', '').strip()
    encoded_auth = ''
    if github_token:
        encoded_auth = base64.b64encode(('x-access-token:' + github_token).encode()).decode()
        # Supply authorization only to GitHub via a child-process environment, never the URL/config file.
        git_env['GIT_CONFIG_COUNT'] = '1'
        git_env['GIT_CONFIG_KEY_0'] = 'http.https://github.com/.extraheader'
        git_env['GIT_CONFIG_VALUE_0'] = 'AUTHORIZATION: basic ' + encoded_auth
    def run_git(arguments):
        result = subprocess.run(['git', *arguments], env=git_env, capture_output=True, text=True)
        if result.returncode:
            details = result.stderr
            for secret in (github_token, encoded_auth):
                if secret:
                    details = details.replace(secret, '[redacted]')
            if 'Could not resolve host' in details:
                raise RuntimeError('GitHub DNS lookup failed. Enable Kaggle Internet or use SOURCE_DIR.\n' + details) from None
            raise RuntimeError(
                'Git failed. For a private repository, enable GITHUB_TOKEN with read access to this repository.\n'
                + details
            ) from None
        return result.stdout.strip()
    try:
        if (PROJECT_ROOT / '.git').is_dir():
            if run_git(['-C', str(PROJECT_ROOT), 'remote', 'get-url', 'origin']) != REPO_URL:
                raise RuntimeError('Existing checkout uses another repository. Start a fresh session.')
            run_git(['-C', str(PROJECT_ROOT), 'pull', '--ff-only'])
        elif PROJECT_ROOT.exists():
            raise RuntimeError('Incomplete source folder exists. Restart the Kaggle session before retrying the clone.')
        else:
            run_git(['clone', '--depth', '1', REPO_URL, str(PROJECT_ROOT)])
    finally:
        # The token is not retained in a Git URL, command argument, or authorization config on disk.
        git_env.clear()
        github_token = encoded_auth = ''
if not (PROJECT_ROOT / MARKER).is_file():
    raise RuntimeError('The repository does not contain the expected Step 05 module.')
os.chdir(PROJECT_ROOT)
print('Project root:', PROJECT_ROOT)
# Create .env in the private /tmp checkout, outside Kaggle saved outputs.
ENV_FILE = PROJECT_ROOT / '.env'
def dotenv_value(value):
    return "'" + value.replace('\\', '\\\\').replace("'", "\\'") + "'"
with ENV_FILE.open('w', encoding='utf-8') as handle:
    os.chmod(ENV_FILE, 0o600)
    for name in sorted(loaded_secret_names):
        if name in ENV_VARIABLE_NAMES:
            handle.write(name + '=' + dotenv_value(os.environ[name]) + '\n')
print('Private environment file created; values are hidden.')


In [ ]:
# Cell 4 — Install the checked training stack without replacing Kaggle's Torch/CUDA packages.
import subprocess
import sys
from collections import deque
from importlib.metadata import version, PackageNotFoundError

if not Path('/kaggle/working').is_dir():
    raise RuntimeError('Run this GPU dependency installation in Kaggle, not on the local computer.')
if any(name in sys.modules for name in ('torch', 'unsloth', 'transformers', 'peft', 'trl')):
    raise RuntimeError('Restart the Kaggle session before installing packages into an already imported training stack.')
preserved_versions = {}
for name in ('torch', 'torchvision', 'torchaudio', 'triton'):
    try:
        preserved_versions[name] = version(name)
    except PackageNotFoundError:
        if name == 'torch':
            raise RuntimeError('Select a Kaggle GPU image with CUDA Torch preinstalled.') from None
preserved_pins = [f'{name}=={release}' for name, release in preserved_versions.items()]
# The resolver must honor these pins; never use --no-deps or force a CUDA stack upgrade.
check_dns('pypi.org')

# Published Unsloth 2026.10.3 requires TRL <=1.13.0.
# TRL 1.13.0 requires Datasets >=4.7.0; Unsloth requires Datasets <5.
KAGGLE_PACKAGES = [
    'unsloth==2026.10.3',
    'unsloth_zoo==2026.10.6',
    'transformers==5.17.0',
    'trl==1.13.0',
    'datasets==4.7.0',
    'accelerate==1.15.0',
    'peft==0.21.2',
    'bitsandbytes==0.50.2',
    'wandb==0.30.0',
    'python-dotenv>=1.0.0',
    'pydantic>=2.7.0',
    'tqdm>=4.66.0',
    'matplotlib>=3.8.0',
    'typesafe-sdk>=0.7.0',
]
# Stream pip output through the notebook and include its final lines in any failure.
# CalledProcessError alone only reports an exit status; it does not explain the cause.
print('Python:', sys.version.split()[0], '| executable:', sys.executable, flush=True)
print('Preserving installed CUDA stack:', preserved_versions, flush=True)
pip_command = [
    sys.executable, '-m', 'pip', 'install',
    '--disable-pip-version-check', '--progress-bar', 'off',
    '--upgrade-strategy', 'only-if-needed', *preserved_pins, *KAGGLE_PACKAGES,
]
pip_tail = deque(maxlen=80)
with subprocess.Popen(
    pip_command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
) as process:
    for line in process.stdout:
        print(line, end='', flush=True)
        pip_tail.append(line)
    return_code = process.wait()
if return_code:
    raise RuntimeError(
        f'Kaggle dependency installation failed (pip exit code {return_code}). '
        'Do not run the model/trainer cells. Share the ERROR lines below so the '
        'failing package or dependency conflict can be identified.\n\n'
        + ''.join(pip_tail)
    )
for name, previous in preserved_versions.items():
    if version(name) != previous:
        raise RuntimeError(f'{name} changed unexpectedly. Start a fresh Kaggle GPU session.')
print('Dependency installation completed; existing CUDA package versions were preserved.', flush=True)
# If resolution fails, keep the printed ERROR lines; do not layer arbitrary Torch upgrades.
# Do not install root requirements.txt afterward: its training pins differ from this profile.
# Installation guide: https://unsloth.ai/docs/get-started/install/pip-install


In [ ]:
# Cell 5 — Import Unsloth first, check visible GPUs, then authenticate without prompts.
import os
import sys
import json
import inspect
from importlib import import_module, reload
from importlib.metadata import version

if 'unsloth' not in sys.modules and any(name in sys.modules for name in ('transformers', 'peft', 'trl')):
    raise RuntimeError('An older training stack is already imported. Start a fresh runtime and run from the top.')
from unsloth import FastLanguageModel
import torch
from trl import SFTConfig, SFTTrainer
from dotenv import load_dotenv
load_dotenv(ENV_FILE, override=False)
from huggingface_hub import login
check_dns('huggingface.co')
login(token=hf_token, add_to_git_credential=False)
GPU_COUNT = torch.cuda.device_count()
if GPU_COUNT < 1:
    raise RuntimeError('Enable GPU T4 x2 and restart the Kaggle session.')
print('Training GPUs:', GPU_COUNT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
fine_tuning = reload(import_module('src.steps.05_fine_tuning.fine_tuning'))
required_helpers = ('apply_lora', 'render_training_prompt', 'build_hub_checkpoint_callback', 'download_checkpoint_from_hub')
missing = [name for name in required_helpers if not callable(getattr(fine_tuning, name, None))]
if missing:
    raise RuntimeError(f'Update the checkout: Step 05 is missing {missing}.')
report_results = import_module('src.steps.04_evaluation').report_results
runtime_versions = {}
for package in ('torch', 'unsloth', 'unsloth_zoo', 'transformers', 'trl', 'peft', 'accelerate', 'datasets', 'bitsandbytes', 'wandb', 'tokenizers', 'pydantic', 'huggingface_hub', 'triton'):
    installed = version(package)
    runtime_versions[package] = installed
    loaded = getattr(sys.modules.get(package), '__version__', installed)
    if str(loaded) != installed:
        raise RuntimeError(f'{package} changed after import. Start a fresh runtime.')
    print(f'{package}: {installed}')
print('Trainer signature:', inspect.signature(SFTTrainer.__init__))
fields = getattr(SFTConfig, '__dataclass_fields__', {})
if not {'max_length', 'completion_only_loss'} <= set(fields):
    raise RuntimeError('Unexpected TRL API. Rerun the notebook installation cell in a fresh runtime.')
if not {'max_steps', 'eval_strategy'} <= set(inspect.signature(fine_tuning.build_training_arguments).parameters):
    raise RuntimeError('Update the GitHub checkout or attached source to the Kaggle Step 05 version.')
if 'step_revisions' not in inspect.signature(fine_tuning.build_hub_checkpoint_callback).parameters:
    raise RuntimeError('Update the source checkout to support step-numbered checkpoint revisions.')
if not (Path(fine_tuning.__file__).parent / 'train.py').is_file():
    raise RuntimeError('Update the source checkout: the distributed training worker is missing.')
print('Fine-tuning implementation:', fine_tuning.__file__)

In [ ]:
# Cell 6 — Noninteractive feature switches and destinations; edit these values for your account.
USE_WANDB = True  # Keep experiment tracking enabled.
HUB_CHECKPOINTS = True
PUSH_TO_HUB = True
HF_PRIVATE = True  # New model repositories are private unless you change this.
RUN_BASELINE = False
RUN_POST_EVAL = True  # Evaluate held-out examples and generate results before publication.
HF_MODEL_REPO = 'md-zain/resume-parser'  # [HF_USERNAME/REPO_NAME]
HF_DATASET_REPO = 'md-zain/resume-parser-dataset'  # [HF_DATASET_REPO]
DATASET_DIR = None  # [DATASET_PATH]: optional '/kaggle/input/YOUR-DATASET/curated'
# Local data must provide train.jsonl, validation.jsonl, test.jsonl with resume_text/target_json/resume_id/category.
if USE_WANDB and not os.getenv('WANDB_API_KEY'):
    raise RuntimeError('Enable WANDB_API_KEY in Kaggle Add-ons → Secrets for this notebook before training.')
print('W&B tracking:', 'enabled' if USE_WANDB else 'disabled')
print('HF model repo:', HF_MODEL_REPO)
print('HF dataset repo:', HF_DATASET_REPO)


In [ ]:
# Cell 7 — config
# Post-evaluation covers every retained validation resume; no sample limit.
MODEL_ID = fine_tuning.DEFAULT_MODEL_ID
LOAD_IN_4BIT = True
WANDB_ENTITY = os.getenv('WANDB_ENTITY')
WANDB_PROJECT = os.getenv('WANDB_PROJECT', 'resume-parser')
print('Base model:', MODEL_ID)
print('Dataset source:', DATASET_DIR or HF_DATASET_REPO)

### One epoch and automatic recovery

The run covers every retained clean training example once. No sample, step or notebook time cap is applied. Before loading model weights, the tokenizer filters complete examples above the native context window, reports the exclusions and sizes the context from the remaining records. Target JSON is never truncated. The dataset-summary cell shows the exact counts that training and validation will use.

The latest complete checkpoint resumes automatically with its original manifest and run ID. In a fresh session, set `RESUME_HUB_REVISION` to a confirmed SHA/experiment branch, or attach prior outputs with `RESUME_CHECKPOINT_DIR`. Recovery requires the same model, data, settings, GPU count and package versions. **Start a new session/output directory for this new full-data configuration; do not resume the previous 496-example experiment.**


In [ ]:
# Cell 8 — Configure one epoch and recover complete checkpoints without interactive input.
from datetime import datetime, timezone
from uuid import uuid4

HYPERPARAMETERS = {
    'learning_rate': 2e-4, 'epochs': 1,
    'max_steps': -1,  # Let epochs determine the training length.
    'per_device_batch_size': 1, 'per_device_eval_batch_size': 1,
    'gradient_accumulation_steps': 16 // GPU_COUNT, 'max_seq_length': None,  # Sized from all examples.
    'lora_r': 16, 'lora_alpha': 16, 'lora_dropout': 0,
    'target_modules': list(fine_tuning.DEFAULT_TARGET_MODULES),
    'optim': 'paged_adamw_8bit', 'evaluate_during_training': True,
    'warmup_ratio': 0.03, 'lr_scheduler_type': 'cosine',
    'save_steps': 50, 'save_total_limit': 2, 'eval_steps': 50,
    'max_train_samples': None, 'seed': 42,  # Use every eligible training example.
}
PROMPT_STYLE = 'compact'
RUN_TAG = 'run'  # Short checkpoint branch: run-YYYYMMDDTHHMMSSZ-<unique id>.
RUN_ID = None  # Optional identity check when recovering; otherwise read from the checkpoint manifest.
RESUME_HUB_REVISION = None  # Confirmed full checkpoint SHA or original experiment branch.
RESUME_CHECKPOINT_DIR = None  # Optional '/kaggle/input/YOUR-PREVIOUS-OUTPUT/outputs'.
OUTPUT_DIR = Path('/kaggle/working/outputs')
ADAPTER_DIR = Path('/kaggle/working/lora_adapter')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def latest_complete_checkpoint(root):
    root = Path(root)
    candidates = [root] if root.name.startswith('checkpoint-') else [
        path for path in root.glob('checkpoint-*') if path.is_dir() and path.name[11:].isdigit()
    ]
    candidates.sort(key=lambda path: int(path.name[11:]) if path.name[11:].isdigit() else -1, reverse=True)
    for candidate in candidates:
        try:
            # Reuse the module's full-state validation, including adapter and RNG files.
            step = fine_tuning._checkpoint_step(candidate)
            if candidate.name != f'checkpoint-{step}':
                raise ValueError('Directory and Trainer step disagree.')
            return candidate
        except (ValueError, KeyError, OSError):
            print('Skipping incomplete checkpoint:', candidate.name)
    return None

if RESUME_HUB_REVISION and RESUME_CHECKPOINT_DIR:
    raise ValueError('Choose either Hub recovery or attached checkpoint recovery.')
local_checkpoint = latest_complete_checkpoint(OUTPUT_DIR)
if local_checkpoint and (RESUME_HUB_REVISION or RESUME_CHECKPOINT_DIR):
    raise ValueError('A local checkpoint already exists. Remove the external recovery setting to resume it.')
if RESUME_HUB_REVISION:
    local_checkpoint = fine_tuning.download_checkpoint_from_hub(
        OUTPUT_DIR, repo_id=HF_MODEL_REPO, revision=RESUME_HUB_REVISION,
    )
elif RESUME_CHECKPOINT_DIR:
    source_checkpoint = latest_complete_checkpoint(RESUME_CHECKPOINT_DIR)
    if source_checkpoint is None:
        raise ValueError('No complete checkpoint found at RESUME_CHECKPOINT_DIR.')
    source_manifest = source_checkpoint / 'run_manifest.json'
    if not source_manifest.is_file():
        source_manifest = source_checkpoint.parent / 'run_manifest.json'
    if not source_manifest.is_file():
        raise ValueError('The attached checkpoint requires its original run_manifest.json.')
    root_manifest = OUTPUT_DIR / 'run_manifest.json'
    if root_manifest.exists() and root_manifest.read_bytes() != source_manifest.read_bytes():
        raise ValueError('Output directory contains another run manifest. Use a fresh session.')
    local_checkpoint = OUTPUT_DIR / source_checkpoint.name
    if local_checkpoint.exists():
        raise ValueError('A partial local checkpoint exists at this step. Use a fresh session before recovery.')
    shutil.copytree(source_checkpoint, local_checkpoint)
    shutil.copyfile(source_manifest, root_manifest)
manifest_path = OUTPUT_DIR / 'run_manifest.json'
if local_checkpoint and not manifest_path.is_file():
    checkpoint_manifest = local_checkpoint / 'run_manifest.json'
    if not checkpoint_manifest.is_file():
        raise ValueError('Cannot resume without the original run_manifest.json.')
    shutil.copyfile(checkpoint_manifest, manifest_path)
previous = json.loads(manifest_path.read_text()) if manifest_path.is_file() else None
if local_checkpoint:
    if RUN_ID and RUN_ID != previous['run_id']:
        raise ValueError('RUN_ID does not match the checkpoint manifest.')
    recovered_step = fine_tuning._checkpoint_step(local_checkpoint)
    recovered_state = json.loads((local_checkpoint / 'trainer_state.json').read_text())
    limit_reached = (
        recovered_step >= HYPERPARAMETERS['max_steps'] if HYPERPARAMETERS['max_steps'] > 0
        else float(recovered_state.get('epoch') or 0) >= HYPERPARAMETERS['epochs']
    )
    if limit_reached:
        raise ValueError(
            f'This run already reached step {recovered_step}, its configured training limit. '
            'Use the saved adapter/checkpoint for inference, or start a new experiment in a fresh session.'
        )
    EXPERIMENT_ID = previous['run_id']
else:
    if previous or any(OUTPUT_DIR.iterdir()):
        raise ValueError('Outputs exist without a complete checkpoint. Inspect/recover them or use a fresh session.')
    EXPERIMENT_ID = RUN_ID or f"{RUN_TAG}-{datetime.now(timezone.utc):%Y%m%dT%H%M%SZ}-{uuid4().hex[:8]}"
RESUME_FROM_CHECKPOINT = str(local_checkpoint) if local_checkpoint else None
START_STEP = fine_tuning._checkpoint_step(local_checkpoint) if local_checkpoint else 0
TRAINED_STEP = START_STEP
RUN_NAME = f"resume-parser-step-{START_STEP}-epoch-{HYPERPARAMETERS['epochs']}-{EXPERIMENT_ID}"
# Trainer and post-evaluation use the same W&B run identity.
os.environ['WANDB_RUN_ID'] = EXPERIMENT_ID
os.environ['WANDB_RESUME'] = 'allow'
print('Run ID:', EXPERIMENT_ID)
print('Resuming:', RESUME_FROM_CHECKPOINT or 'new run')
print('Output dir:', OUTPUT_DIR)
print(json.dumps(HYPERPARAMETERS, indent=2))


## 1. Download, validate, and prepare the splits

The test split is downloaded only to reserve its resume identities/text. It supplies no training targets. Training excludes all held-out overlap; duplicate texts are deduplicated, and groups with conflicting targets are excluded with counts. Original dataset files are preserved.

In [ ]:
# Cell 9 — Load the project's curated JSONL files; keep raw resumes out of saved Kaggle outputs.
if DATASET_DIR:
    data_root = Path(DATASET_DIR)
    splits = {name: data_root / f'{name}.jsonl' for name in ('train', 'validation', 'test')}
    if any(not path.is_file() for path in splits.values()):
        raise FileNotFoundError('DATASET_DIR needs train.jsonl, validation.jsonl, and test.jsonl.')
else:
    splits = fine_tuning.download_curated_splits(
        splits=('train', 'validation', 'test'), repo_id=HF_DATASET_REPO,
        curated_dir=PROJECT_ROOT / '.data/curated', force=True,
    )
raw_train_records = fine_tuning.load_records(splits['train'])
raw_validation_records = fine_tuning.load_records(splits['validation'])
test_records = fine_tuning.load_records(splits['test'])
train_records, validation_records, preparation_report = fine_tuning.prepare_training_records(
    raw_train_records, raw_validation_records, test_records=test_records,
)
print(json.dumps(preparation_report, indent=2))


In [ ]:
# Cell 10 — Inspect both GPUs; no model weights are loaded in this kernel yet.
import gc
subprocess.run(['nvidia-smi'], check=True)
if torch.cuda.device_count() != GPU_COUNT:
    raise RuntimeError('GPU visibility changed. Start a fresh session.')
for index in range(GPU_COUNT):
    print(index, torch.cuda.get_device_name(index),
          round(torch.cuda.get_device_properties(index).total_memory / 2**30, 2), 'GiB')
print('Training precision: fp16=True, bf16=False')


## 2. Filter oversized examples before loading weights

Only tokenizer/config files are loaded here. Length includes the compact prompt, Qwen chat formatting and complete target JSON. Examples above the native context window are excluded from both training and validation; all remaining records are retained without sampling. The original split files are unchanged.


In [ ]:
# Cell 11 — Filter complete examples above the native context; never truncate JSON.
from transformers import AutoConfig, AutoTokenizer
from functools import partial
prompt_fn = partial(fine_tuning.render_training_prompt, style=PROMPT_STYLE)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=hf_token)
model_config = AutoConfig.from_pretrained(MODEL_ID, token=hf_token)
if not tokenizer.chat_template or tokenizer.eos_token_id is None:
    raise ValueError('The model tokenizer requires its Qwen chat template and EOS token.')
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'
native_context = int(model_config.max_position_embeddings)
fit_train_records, train_lengths = fine_tuning.filter_records_by_length(
    train_records, tokenizer, max_length=native_context, policy='drop', prompt_fn=prompt_fn,
)
fit_validation_records, validation_lengths = fine_tuning.filter_records_by_length(
    validation_records, tokenizer, max_length=native_context, policy='drop', prompt_fn=prompt_fn,
)
if any('max_kept_length' not in report for report in (train_lengths, validation_lengths)):
    raise RuntimeError('Update fine_tuning.py and reload it: the retained-length reporting helper is missing.')
print('Native context window:', native_context, 'tokens')
print('Training length report:', json.dumps(train_lengths, indent=2))
print('Validation length report:', json.dumps(validation_lengths, indent=2))
longest_retained = max(train_lengths['max_kept_length'], validation_lengths['max_kept_length'])
HYPERPARAMETERS['max_seq_length'] = min(native_context, max(256, ((longest_retained + 159) // 128) * 128))
MAX_NEW_TOKENS = HYPERPARAMETERS['max_seq_length']
print('Selected context for retained examples:', HYPERPARAMETERS['max_seq_length'])


## 3. Prepare all retained examples

The GPU workers attach rank-16 LoRA adapters with `use_gradient_checkpointing="unsloth"`. Training uses conversational prompt/completion columns with loss only on the assistant JSON. Validation receives the same formatting without updating weights.


In [ ]:
# Cell 12 — LoRA settings; adapters are attached once inside each GPU worker.
import hashlib
from collections import Counter
print('LoRA rank:', HYPERPARAMETERS['lora_r'])
print('Effective batch size:', HYPERPARAMETERS['per_device_batch_size']
      * HYPERPARAMETERS['gradient_accumulation_steps'] * GPU_COUNT)


In [ ]:
# Cell 13 — Build datasets from all records remaining after the native-context filter.
train_dataset, eval_dataset = fine_tuning.build_datasets(
    fit_train_records, fit_validation_records, tokenizer=tokenizer,
    max_train=None, max_seq_length=HYPERPARAMETERS['max_seq_length'],
    seed=HYPERPARAMETERS['seed'], test_records=test_records, prompt_fn=prompt_fn,
)
if len(train_dataset) != len(fit_train_records) or len(eval_dataset) != len(fit_validation_records):
    raise RuntimeError('Prepared counts changed after filtering. Inspect the preparation report before training.')
NUM_TRAIN_RESUMES = len(train_dataset)
REVISION = EXPERIMENT_ID
selected_records = fine_tuning.select_training_records(fit_train_records, None, seed=HYPERPARAMETERS['seed'])
print('Selected category counts:', dict(Counter(r.get('category') for r in selected_records)))


In [ ]:
# Cell 13a — Confirm exact training/validation usage before loading model weights.
import pandas as pd
from IPython.display import display

dataset_summary = pd.DataFrame({
    'Stage': [
        'Source split rows',
        'Removed by duplicate/conflict/held-out cleanup',
        'Clean candidates',
        f'Excluded above {native_context:,} tokens',
        'Final records used',
    ],
    'Training': [
        len(raw_train_records), len(raw_train_records) - len(train_records),
        len(train_records), train_lengths['over_max_length'], len(train_dataset),
    ],
    'Validation': [
        len(raw_validation_records), len(raw_validation_records) - len(validation_records),
        len(validation_records), validation_lengths['over_max_length'], len(eval_dataset),
    ],
}).set_index('Stage')
display(dataset_summary)
print(f"Training: {len(train_dataset):,} resumes for exactly {HYPERPARAMETERS['epochs']} epoch.")
print(f"Validation: all {len(eval_dataset):,} retained resumes per loss check and final JSON evaluation.")
print(f"GPUs: {GPU_COUNT} | Context: {HYPERPARAMETERS['max_seq_length']:,} tokens | Test data: reserved.")


## 4. Train one epoch with both T4s

`torchrun` creates one process per visible GPU using distributed data parallel training. Each GPU holds its own 4-bit replica and receives different training samples. The notebook kernel keeps no training model in GPU memory. Two T4s improve throughput; they do not provide one combined 32 GB memory allocation.

Full validation loss is logged in W&B before training, at each 50-step checkpoint and at the final step. The final checkpoint is saved even when the epoch ends between checkpoint intervals. Adapter export happens before slower post-evaluation. Optional baseline JSON evaluation and final JSON scoring use the entire same retained validation split; checkpoint loss checks do not run JSON generation.


In [ ]:
# Cell 14 — Record reproducible inputs and prepare the distributed worker job.
manifest = {
    'backend': 'unsloth-ddp', 'prompt_style': PROMPT_STYLE,
    'run_id': EXPERIMENT_ID, 'model_id': MODEL_ID, 'hyperparameters': HYPERPARAMETERS,
    'model_repo': HF_MODEL_REPO, 'dataset_repo': HF_DATASET_REPO, 'gpu_count': GPU_COUNT,
    'runtime_versions': runtime_versions,
    'training_precision': {'bf16': False, 'fp16': True},
    'prompt_sha256': hashlib.sha256(prompt_fn('STATIC_PROMPT_MARKER').encode('utf-8')).hexdigest(),
    'schema_sha256': hashlib.sha256((PROJECT_ROOT / 'src/schema/resume_output.py').read_bytes()).hexdigest(),
    'dataset_revision': os.getenv('HF_DATASET_REVISION', 'main'),
    'split_sha256': {name: hashlib.sha256(path.read_bytes()).hexdigest() for name, path in splits.items()},
    'train_resume_ids': [r['resume_id'] for r in selected_records],
    'validation_resume_ids': [r['resume_id'] for r in fit_validation_records],
    'length_filter': {'policy': 'drop', 'max_length': native_context},
    'preparation': preparation_report, 'train_lengths': train_lengths, 'validation_lengths': validation_lengths,
}
if RESUME_FROM_CHECKPOINT:
    previous = json.loads(manifest_path.read_text())
    for key in ('backend', 'prompt_style', 'run_id', 'model_id', 'model_repo', 'dataset_repo',
                'hyperparameters', 'gpu_count', 'split_sha256', 'train_resume_ids', 'validation_resume_ids', 'length_filter', 'runtime_versions',
                'training_precision', 'prompt_sha256', 'schema_sha256'):
        if previous.get(key) != manifest[key]:
            raise ValueError(f'Resume configuration changed: {key}. Use a fresh output directory/session.')
else:
    manifest_path.write_text(json.dumps(manifest, indent=2), encoding='utf-8')
job_dir = PRIVATE_ROOT / 'training-jobs' / EXPERIMENT_ID
job_dir.mkdir(parents=True, exist_ok=True)
train_dataset.save_to_disk(str(job_dir / 'train'))
eval_dataset.save_to_disk(str(job_dir / 'validation'))
job = {
    'run_id': EXPERIMENT_ID, 'run_name': RUN_NAME, 'model_id': MODEL_ID,
    'model_repo': HF_MODEL_REPO, 'revision': REVISION, 'hf_private': HF_PRIVATE,
    'hyperparameters': HYPERPARAMETERS, 'gpu_count': GPU_COUNT,
    'train_dataset': str(job_dir / 'train'), 'validation_dataset': str(job_dir / 'validation'),
    'train_count': len(train_dataset), 'validation_count': len(eval_dataset),
    'output_dir': str(OUTPUT_DIR), 'adapter_dir': str(ADAPTER_DIR),
    'resume_from_checkpoint': RESUME_FROM_CHECKPOINT,
    'use_wandb': USE_WANDB, 'wandb_entity': WANDB_ENTITY, 'wandb_project': WANDB_PROJECT,
    'hub_checkpoints': HUB_CHECKPOINTS,
}
job_path = job_dir / 'job.json'
job_path.write_text(json.dumps(job, indent=2), encoding='utf-8')
print('One epoch over', len(train_dataset), 'training records on', GPU_COUNT, 'GPUs')
print('Full validation:', len(eval_dataset), 'records before training, every 50 updates, and at the end')


In [ ]:
# Cell 15 — Launch GPU workers; no manual sample, step or time cap.
if RUN_BASELINE and not RESUME_FROM_CHECKPOINT:
    def evaluate_base_model():
        baseline_model, baseline_tokenizer = fine_tuning.load_model_and_tokenizer(
            MODEL_ID, max_seq_length=HYPERPARAMETERS['max_seq_length'],
        )
        return fine_tuning.evaluate_records(
            baseline_model, baseline_tokenizer, fit_validation_records,
            model_name=f'baseline:{MODEL_ID}', limit=None, max_new_tokens=MAX_NEW_TOKENS,
            max_seq_length=HYPERPARAMETERS['max_seq_length'], prompt_fn=prompt_fn,
            seed=HYPERPARAMETERS['seed'],
            output_dir=PROJECT_ROOT / '.data/pre-evaluation' / EXPERIMENT_ID,
        )
    baseline_report = evaluate_base_model()
    print('Baseline:', json.dumps(baseline_report.metrics, indent=2))
    fine_tuning.free_memory()
subprocess.run([
    sys.executable, '-m', 'torch.distributed.run', '--standalone',
    '--nproc_per_node', str(GPU_COUNT), str(Path(fine_tuning.__file__).parent / 'train.py'),
    '--job', str(job_path),
], cwd=str(PROJECT_ROOT), check=True)
training_result = json.loads((OUTPUT_DIR / 'training_result.json').read_text())
if training_result['run_id'] != EXPERIMENT_ID or training_result['epoch'] < 1 - 1e-6:
    raise RuntimeError('The expected one-epoch run did not complete; inspect or resume its checkpoint.')
TRAINED_STEP = training_result['global_step']
RUN_NAME = training_result['run_name']
print('Completed:', RUN_NAME)
print('Confirmed checkpoint commits:')
for uploaded in training_result['checkpoint_commits']:
    print(uploaded.get('step_revision', uploaded['revision']), uploaded['commit_sha'], uploaded['commit_url'])


In [ ]:
# Cell 16 — Show peak GPU memory recorded by each training worker.
for peak in training_result['peak_memory']:
    print(f"GPU {peak['gpu']}: allocated {peak['allocated_gib']:.2f} GiB; reserved {peak['reserved_gib']:.2f} GiB")


In [ ]:
# Cell 17 — Workers saved the adapter before exiting; verify its location.
adapter_dir = ADAPTER_DIR
if not (adapter_dir / 'adapter_config.json').is_file() or not list(adapter_dir.glob('adapter_model*.safetensors')):
    raise RuntimeError('The completed training worker did not export its LoRA adapter.')
print('Saved adapter to:', adapter_dir)


## 5. Post-training evaluation on all retained validation records

Every validation record remaining after cleanup and the native-context filter is evaluated with JSON generation, without further sampling. The dataset-summary cell reports all exclusions before training. Full validation loss was already measured during training. The untouched test split remains reserved. Tables show JSON/schema validity, programmatic and semantic extraction scores, overall score and field breakdowns. Semantic scoring requires `TYPESAFE_API_KEY`; missing semantic scores stay explicitly unscored.

The model cell reloads the exported adapter on GPU 0 after training workers exit. Final publication to `main` follows evaluation. To evaluate after a session reset, restore source/dependencies, prepare validation data and select the saved adapter first; do not rerun training.


In [ ]:
# Cell 18 — evaluation model
# Reuse the trained model or reload a saved adapter for evaluation.
from pathlib import Path
from unsloth import FastLanguageModel
import torch

EVAL_ADAPTER_DIR = None  # Optional local adapter/checkpoint folder, including a surviving checkpoint-100.
EVAL_ADAPTER_REVISION = None  # Optional Hub branch/SHA; overrides the current model and local adapter.
EVAL_ADAPTER_REPO = HF_MODEL_REPO
ADAPTER_FILES = [
    'adapter_config.json', 'adapter_model*.safetensors', 'adapter_model*.bin',
    'tokenizer.json', 'tokenizer_config.json', 'special_tokens_map.json',
    'added_tokens.json', 'vocab.json', 'merges.txt', 'chat_template.jinja', 'chat_templates/*.jinja',
]

def has_saved_adapter(folder):
    return (folder / 'adapter_config.json').is_file() and bool(
        list(folder.glob('adapter_model*.safetensors')) or list(folder.glob('adapter_model*.bin'))
    )

def resolve_saved_adapter():
    if EVAL_ADAPTER_DIR and EVAL_ADAPTER_REVISION:
        raise ValueError('Choose a local adapter folder or a Hub revision, not both.')
    if EVAL_ADAPTER_DIR:
        candidate = Path(EVAL_ADAPTER_DIR)
        if not has_saved_adapter(candidate):
            raise FileNotFoundError('EVAL_ADAPTER_DIR needs adapter_config.json and adapter weights.')
        return candidate
    if not EVAL_ADAPTER_REVISION:
        candidate = Path(ADAPTER_DIR)
        if has_saved_adapter(candidate):
            return candidate
        # Local checkpoints can survive even if the remote repository has been deleted.
        candidates = [p for p in Path(OUTPUT_DIR).glob('checkpoint-*') if p.name[11:].isdigit()]
        for candidate in sorted(candidates, key=lambda p: int(p.name[11:]), reverse=True):
            try:
                fine_tuning._checkpoint_step(candidate)
            except (ValueError, KeyError, OSError):
                continue
            if has_saved_adapter(candidate):
                return candidate
    revision = EVAL_ADAPTER_REVISION or globals().get('REVISION')
    if not revision:
        raise RuntimeError('No local trained adapter/checkpoint is available. Select an existing uploaded run revision.')
    from huggingface_hub import snapshot_download
    snapshot = Path(snapshot_download(
        repo_id=EVAL_ADAPTER_REPO, revision=revision, token=os.getenv('HF_TOKEN'),
        allow_patterns=ADAPTER_FILES + ['checkpoint/' + pattern for pattern in ADAPTER_FILES],
    ))
    candidate = snapshot if has_saved_adapter(snapshot) else snapshot / 'checkpoint'
    if not has_saved_adapter(candidate):
        raise RuntimeError('The selected Hub revision contains no trained adapter. Select the uploaded run branch/SHA.')
    print('Evaluation adapter snapshot SHA:', snapshot.name)
    return candidate


def get_evaluation_model():
    explicit_source = bool(EVAL_ADAPTER_DIR or EVAL_ADAPTER_REVISION)
    active_trainer = globals().get('trainer')
    trained_model = getattr(active_trainer, 'model', None)
    trained_tokenizer = globals().get('tokenizer')
    if not explicit_source and trained_model is not None and trained_tokenizer is not None:
        if not getattr(trained_model, 'peft_config', None):
            raise RuntimeError('Evaluation requires the trained LoRA adapter, not a bare base model.')
        return trained_model, trained_tokenizer
    source_key = (str(EVAL_ADAPTER_DIR or ADAPTER_DIR), EVAL_ADAPTER_REPO, EVAL_ADAPTER_REVISION or REVISION)
    if (globals().get('_evaluation_source_key') == source_key
            and globals().get('evaluation_model') is not None
            and globals().get('evaluation_tokenizer') is not None):
        return evaluation_model, evaluation_tokenizer
    adapter_path = resolve_saved_adapter()
    # Release old inference references before loading the exported adapter on GPU 0.
    active_trainer = trained_model = trained_tokenizer = None
    for name in ('trainer', 'model', 'tokenizer', 'callback', 'evaluation_model', 'evaluation_tokenizer'):
        globals().pop(name, None)
    fine_tuning.free_memory()
    recovered_model, recovered_tokenizer = fine_tuning.load_model_and_tokenizer(
        str(adapter_path), load_in_4bit=True, max_seq_length=HYPERPARAMETERS['max_seq_length'],
    )
    if not getattr(recovered_model, 'peft_config', None):
        raise RuntimeError('The trained adapter was not attached. Refusing to evaluate an untuned base model.')
    FastLanguageModel.for_inference(recovered_model)
    globals()['evaluation_model'] = recovered_model
    globals()['evaluation_tokenizer'] = recovered_tokenizer
    globals()['_evaluation_source_key'] = source_key
    print('Reloaded trained adapter for evaluation:', adapter_path)
    return recovered_model, recovered_tokenizer


In [ ]:
# Cell 19 — quick inference
# Use synthetic text for a quick generation check without printing private resumes.
evaluation_model, evaluation_tokenizer = get_evaluation_model()
quick_item = fine_tuning.EvaluationItem(
    resume_id='synthetic-smoke-check',
    resume_text='Alex Example is a Python developer. Worked at Example Company from 2021 to 2024. Skills: Python, SQL.',
    target_json={},
)
callback = fine_tuning.make_generation_callback(
    evaluation_model, evaluation_tokenizer, model_name='unsloth-qlora', prompt_fn=prompt_fn,
    max_seq_length=HYPERPARAMETERS['max_seq_length'], max_new_tokens=MAX_NEW_TOKENS,
)
quick_output = callback(quick_item)
print(quick_output.raw_output)
try:
    quick_json = json.loads(quick_output.raw_output)
    from src.schema.resume_output import ResumeOutput
    ResumeOutput.model_validate(quick_json)
    print('Synthetic output is valid JSON matching ResumeOutput.')
except (ValueError, TypeError):
    print('Synthetic output is incomplete or invalid. Inspect the output and training length coverage.')

In [ ]:
# Cell 20 — evaluate
# Generate and score JSON for all retained validation records; save anonymous score tables.
if RUN_POST_EVAL:
    if 'fit_validation_records' not in globals() or not fit_validation_records:
        raise RuntimeError('Prepare the held-out validation records before evaluation. Do not rerun training.')
    evaluation_model, evaluation_tokenizer = get_evaluation_model()
    evaluated_adapter_source = str(resolve_saved_adapter().resolve())
    print('Post-evaluation resumes:', len(fit_validation_records))
    if not os.getenv('TYPESAFE_API_KEY'):
        print('TYPESAFE_API_KEY is unset: programmatic checks still run; unavailable semantic scores stay unscored.')
    report = fine_tuning.evaluate_records(
        evaluation_model, evaluation_tokenizer, fit_validation_records,
        model_name=f'finetuned:{MODEL_ID}', limit=None,
        max_new_tokens=MAX_NEW_TOKENS, max_seq_length=HYPERPARAMETERS['max_seq_length'],
        prompt_fn=prompt_fn, seed=HYPERPARAMETERS['seed'],
        # Detailed results contain resume text/targets; keep them outside saved Kaggle outputs.
        output_dir=PROJECT_ROOT / '.data/post-evaluation' / EXPERIMENT_ID,
    )
    import pandas as pd
    from IPython.display import display
    from collections import defaultdict
    metrics = {key: value for key, value in report.metrics.items() if key != 'selected_resume_ids'}
    ordered_results = sorted(report.results, key=lambda result: result.item.resume_id)
    evaluation_table = pd.DataFrame([
        {
            'case': f'validation-{index:03d}', 'category': result.item.category,
            'status': result.status, 'json_valid': result.json_valid, 'schema_valid': result.schema_valid,
            'programmatic_score': result.programmatic_score, 'semantic_score': result.semantic_score,
            'overall_score': result.overall_score, 'identity_score': result.identity_score,
            'duration_seconds': result.duration_seconds,
        }
        for index, result in enumerate(ordered_results, 1)
    ])
    total = len(ordered_results)
    summary_rows = [
        {'metric': 'Resumes evaluated', 'value': total},
        {'metric': 'JSON valid count', 'value': metrics['json_valid_count']},
        {'metric': 'JSON valid rate', 'value': metrics['json_valid_count'] / total if total else None},
        {'metric': 'Schema valid count', 'value': metrics['schema_valid_count']},
        {'metric': 'Schema valid rate', 'value': metrics['schema_valid_count'] / total if total else None},
        {'metric': 'Fully scored resumes', 'value': metrics['scored_count']},
        {'metric': 'Mean programmatic field score', 'value': metrics['mean_programmatic_score']},
        {'metric': 'Mean semantic field score', 'value': metrics['mean_semantic_score']},
        {'metric': 'Mean overall score among scored resumes', 'value': metrics['mean_overall_score']},
        {'metric': 'Mean identity score', 'value': metrics['mean_identity_score']},
        {'metric': 'Overall score including invalid outputs', 'value': metrics['overall_score_including_invalid_predictions']},
    ]
    evaluation_summary_table = pd.DataFrame(summary_rows)
    field_values = defaultdict(list)
    for result in ordered_results:
        for field in result.score_breakdown:
            if field.get('value') is not None:
                field_values[(field['group'], field['path'])].append(float(field['value']))
    evaluation_field_table = pd.DataFrame([
        {'group': group, 'field': field, 'mean_score': sum(values) / len(values), 'scored_comparisons': len(values)}
        for (group, field), values in sorted(field_values.items())
    ], columns=['group', 'field', 'mean_score', 'scored_comparisons'])
    display(evaluation_summary_table, evaluation_table, evaluation_field_table)
    print('Status counts:', metrics['status_counts'])
    print('Scores/rates use 0–1. Missing scores are unscored; they are not perfect scores or silently replaced with zero.')
    report_dir = OUTPUT_DIR / 'evaluation-after'
    report_dir.mkdir(parents=True, exist_ok=True)
    evaluation_summary_table.to_csv(report_dir / 'summary.csv', index=False)
    evaluation_table.to_csv(report_dir / 'per_resume.csv', index=False)
    evaluation_field_table.to_csv(report_dir / 'per_field.csv', index=False)
    evaluation_summary = {
        'run_id': EXPERIMENT_ID, 'model_id': MODEL_ID, 'model_repo': HF_MODEL_REPO,
        'checkpoint_revision': REVISION, 'adapter_source': evaluated_adapter_source,
        'validation_scope': 'all clean validation records fitting the native context; no sampling',
        'validation_length_filter': validation_lengths,
        'dataset_counts': dataset_summary.to_dict(),
        'max_seq_length': HYPERPARAMETERS['max_seq_length'], 'max_new_tokens': MAX_NEW_TOKENS,
        'semantic_key_available': bool(os.getenv('TYPESAFE_API_KEY')), 'metrics': metrics,
    }
    (report_dir / 'metrics.json').write_text(json.dumps(evaluation_summary, indent=2) + '\n', encoding='utf-8')
    print('Saved anonymous evaluation tables:', report_dir)
    if USE_WANDB:
        import wandb
        evaluation_run = wandb.run
        owns_evaluation_run = evaluation_run is None
        if owns_evaluation_run:
            evaluation_run = wandb.init(
                project=WANDB_PROJECT, entity=WANDB_ENTITY, id=EXPERIMENT_ID,
                name=RUN_NAME, resume='allow', job_type='post-evaluation',
            )
        try:
            evaluation_run.log({
                **{f'post_eval/{key}': value for key, value in metrics.items() if isinstance(value, (int, float))},
                'post_eval/per_resume': wandb.Table(dataframe=evaluation_table),
                'post_eval/summary': wandb.Table(dataframe=evaluation_summary_table),
                'post_eval/per_field': wandb.Table(dataframe=evaluation_field_table),
            })
            evaluation_run.summary['post_eval/status_counts'] = metrics['status_counts']
            evaluation_run.summary['post_eval/validation_scope'] = evaluation_summary['validation_scope']
        finally:
            if owns_evaluation_run:
                evaluation_run.finish()
else:
    print('Post-evaluation skipped by explicit configuration. Set RUN_POST_EVAL=True to produce validation tables.')


### Publish the final evaluated adapter to main

Checkpoint history stays on the unique `run-<UTC timestamp>-<id>` branch. Each checkpoint SHA also has a readable branch such as `checkpoint-step-50-run-<UTC timestamp>-<id>` or `checkpoint-step-100-run-<UTC timestamp>-<id>`. After post-evaluation and W&B result logging, the final inference adapter/tokenizer and aggregate evaluation summary are uploaded to **main**. This cell uses saved files, so it also works after the Trainer has been removed. The HF token creates the new repository automatically when this cell or a checkpoint callback runs.


In [ ]:
# Cell 21 — push
# Publish the evaluated adapter to main from saved files; a Trainer object is not required.
if PUSH_TO_HUB:
    if RUN_POST_EVAL and 'evaluation_summary' not in globals():
        raise RuntimeError('Run post-evaluation before publishing the final adapter to main.')
    from huggingface_hub import HfApi
    from tempfile import TemporaryDirectory
    import shutil
    source_adapter = resolve_saved_adapter()
    if RUN_POST_EVAL and (
        evaluation_summary['run_id'] != EXPERIMENT_ID
        or evaluation_summary['adapter_source'] != str(source_adapter.resolve())
    ):
        raise RuntimeError('The selected adapter/run changed after evaluation. Evaluate this adapter before publication.')
    api = HfApi(token=os.getenv('HF_TOKEN'))
    api.create_repo(repo_id=HF_MODEL_REPO, repo_type='model', private=HF_PRIVATE, exist_ok=True)
    # Stage only inference files and anonymous aggregate metadata, never optimizer state, secrets, or resumes.
    with TemporaryDirectory(dir=PRIVATE_ROOT, prefix='release-') as staging:
        release_dir = Path(staging)
        for pattern in ADAPTER_FILES:
            for source_file in source_adapter.glob(pattern):
                if source_file.is_file():
                    target = release_dir / source_file.relative_to(source_adapter)
                    target.parent.mkdir(parents=True, exist_ok=True)
                    shutil.copy2(source_file, target)
        if not has_saved_adapter(release_dir):
            raise RuntimeError('The release is missing its trained LoRA adapter.')
        if RUN_POST_EVAL:
            (release_dir / 'evaluation_summary.json').write_text(
                json.dumps(evaluation_summary, indent=2) + '\n', encoding='utf-8',
            )
        release_metadata = {
            'run_id': EXPERIMENT_ID, 'base_model': MODEL_ID,
            'checkpoint_revision': REVISION, 'training_hyperparameters': HYPERPARAMETERS,
            'post_evaluation_enabled': RUN_POST_EVAL,
        }
        (release_dir / 'release_metadata.json').write_text(json.dumps(release_metadata, indent=2) + '\n', encoding='utf-8')
        (release_dir / 'README.md').write_text(
            '---\nbase_model: ' + MODEL_ID + '\nlibrary_name: peft\ntags:\n- qlora\n- resume-parsing\n---\n'
            '# Resume Parser\n\n'
            'LoRA adapter for extracting structured resume JSON using the project ResumeOutput layout.\n\n'
            'Load with its 4-bit base model and use the compact extraction prompt from Step 05. '
            'The published files contain adapter weights and tokenizer files, not a merged full model.\n\n'
            'See evaluation_summary.json for measured validation results when enabled. '
            'Those results cover all retained validation records after cleanup and native-context filtering; '
            'they do not establish accuracy on unseen datasets or all resume formats.\n',
            encoding='utf-8',
        )
        commit = api.upload_folder(
            repo_id=HF_MODEL_REPO, repo_type='model', folder_path=str(release_dir), revision='main',
            allow_patterns=ADAPTER_FILES + ['README.md', 'evaluation_summary.json', 'release_metadata.json'],
            commit_message=f'Publish evaluated resume parser adapter ({EXPERIMENT_ID})', run_as_future=False,
        )
    print('Published model:', f'https://huggingface.co/{HF_MODEL_REPO}/tree/main')
    print('Release commit SHA:', commit.oid)
    print('Release commit:', commit.commit_url)
else:
    print('Adapter remains saved locally; publication disabled.')


## If a complete example exceeds available memory

The run uses batch size 1 per GPU, FP16, a 4-bit base model and Unsloth gradient checkpointing. DDP keeps one replica per GPU; memory is not pooled. The context report prints complete-example lengths and counts exclusions above the native context. The dataset-summary cell displays exactly how many records will be used before loading weights.

If long examples cause OOM, restart the Kaggle session. Reduce LoRA rank/alpha to 8 or use the 1.5B model for a separate experiment. Examples exceeding the native context are already filtered with explicit counts and intact target JSON. Retained examples near that limit may still exceed T4 memory; the native-context filter is not a guarantee of GPU fit. Any further context reduction must be reported as an additional dataset exclusion.

Kaggle can interrupt a run when its own session/quota limit is reached. Resume a confirmed full checkpoint with the same data, context, GPU count and settings. Only static source checks were performed locally; installation, distributed training and GPU memory must be verified on Kaggle.


### Values to customize before Save & Run All

- **Source cell:** `REPO_URL` (`[REPO_URL]`); the current project URL is filled in. For private GitHub access, enable `GITHUB_TOKEN` with read access. If GitHub cannot be reached, attach source as a private Kaggle input and set `SOURCE_DIR` to the directory containing `src/`.
- **Secrets cell:** `ENV_VARIABLE_NAMES` (`[LIST OF ENV VARIABLE NAMES]`). Enable `HF_TOKEN` with write access. W&B tracking is enabled by default and requires `WANDB_API_KEY`; `WANDB_PROJECT` and `WANDB_ENTITY` select its destination. Typesafe credentials are optional unless semantic evaluation is enabled. Values are never printed and the generated `.env` stays under `/tmp`.
- **Feature-switch cell:** `HF_MODEL_REPO` (`[HF_USERNAME/REPO_NAME]`), `HF_DATASET_REPO`, and optional `DATASET_DIR` (`[DATASET_PATH]`). The new model repository is `md-zain/resume-parser`; the dataset repository is unchanged. Local curated JSONL files must have `resume_text`, `target_json`, `resume_id`, and `category` fields.
- **Training:** one epoch, every clean train/validation row fitting the native context, automatically sized context and both T4s. Review the dataset-summary cell before training. No notebook sample/step/time cap. Full validation loss runs every 50 steps and at completion; full JSON scoring runs after training. Runtime depends on all example lengths and Kaggle resources.
- **Recovery in a fresh session:** set `RESUME_HUB_REVISION` to a confirmed checkpoint SHA/branch, or attach prior outputs and set `RESUME_CHECKPOINT_DIR`. Existing local complete checkpoints resume automatically. Leave recovery values unset for a new run.

If cloning still reports `Could not resolve host: github.com`, enable **Internet** in Kaggle settings and retry in a fresh session. A token cannot repair DNS. An attached source folder avoids the GitHub clone, but package installation/model/dataset downloads and Hub uploads still require network access. Make sure the latest notebook **and module changes** are available in the chosen repository/source folder.

References: [Kaggle notebook settings](https://www.kaggle.com/docs/notebooks), [Kaggle Secrets implementation](https://github.com/Kaggle/docker-python/blob/main/patches/kaggle_secrets.py), [Unsloth installation](https://unsloth.ai/docs/get-started/install/pip-install). Only static checks were performed locally; installation, GPU memory, background execution, and Hub transfers must be verified on Kaggle.

- **Evaluation recovery:** `EVAL_ADAPTER_DIR` can select a local/attached trained adapter, or `EVAL_ADAPTER_REPO` and `EVAL_ADAPTER_REVISION` can select the uploaded run branch/checkpoint SHA. When unset, the current trained model or local adapter export is used. The trained model stays available for manual evaluation.

- **Post-evaluation and release:** `RUN_POST_EVAL=True` evaluates all retained validation records before the final upload to `main`. Checkpoint snapshots use `checkpoint-step-<actual step>-run-<UTC timestamp>-<unique id>`; the run branch keeps their complete commit history. W&B names include the starting step and one-epoch target while training and the actual completed step afterward, for example `resume-parser-step-100-run-<UTC timestamp>-<unique id>`. W&B uses the same experiment ID for training and post-evaluation logs.
